# Sentiment Analysis - Minecraft Reviews (TensorFlow)
**Nama**: Asep Haryana Saputra

## Project Overview
**Objective**: Build a high-accuracy (>92%) sentiment analysis model.
**Dataset**: 40,000 Minecraft reviews.
**Framework**: TensorFlow / Keras.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import tensorflow as tf
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
try:
    from tensorflow.keras.layers import TextVectorization
except ImportError:
    from tensorflow.keras.layers.experimental.preprocessing import TextVectorization
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, GRU, Dropout, GlobalAveragePooling1D, Input
from tensorflow.keras.models import Sequential, Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")

# Optimize GPU Memory
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)


2026-02-10 01:46:19.931450: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-10 01:46:19.969326: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-10 01:46:20.986171: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow Version: 2.20.0


In [2]:
# Callbacks Definition
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

def get_callbacks(model_name):
    checkpoint = ModelCheckpoint(
        f'{model_name}_best.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
    early_stopping = EarlyStopping(
        monitor='val_accuracy',
        min_delta=0.001,
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
    return [checkpoint, early_stopping, reduce_lr]

## 2. Data Loading & Preprocessing
**Labeling Strategy**:
- 1-2 Stars -> **Negative (0)**
- 3 Stars -> **Neutral (1)**
- 4-5 Stars -> **Positive (2)**

In [3]:
# Load Data
df = pd.read_csv('dataset_raw.csv')
df.dropna(subset=['content'], inplace=True)

# Labeling
def map_sentiment(score):
    if score <= 2: return 0
    elif score == 3: return 1
    else: return 2

df['label'] = df['score'].apply(map_sentiment)

# Clean Text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_content'] = df['content'].apply(clean_text)
print(df['label'].value_counts())

label
2    31359
0     6988
1     1650
Name: count, dtype: int64


In [4]:
# 3. Data Filtering & Splitting
# Remove extremely short reviews (noise)
if 'content' in df.columns:
    df['word_count'] = df['content'].apply(lambda x: len(str(x).split()))
    df_clean = df[df['word_count'] >= 3].copy()
    print(f"Original shape: {df.shape}, Filtered shape: {df_clean.shape}")

    X = df_clean['content'].values
    y = df_clean['label'].values

    # Scheme 1 & 3: 80/20 Split
    X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Scheme 2: 90/10 Split
    X_train_90, X_test_90, y_train_90, y_test_90 = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

    print(f"Scheme 1/3 (80/20) - Train: {X_train_80.shape}, Test: {X_test_80.shape}")
    print(f"Scheme 2 (90/10)   - Train: {X_train_90.shape}, Test: {X_test_90.shape}")
else:
    print("Error: 'content' column not found yet. Ensure Data Loading cell is run first.")

Original shape: (39997, 14), Filtered shape: (32761, 14)
Scheme 1/3 (80/20) - Train: (26208,), Test: (6553,)
Scheme 2 (90/10)   - Train: (29484,), Test: (3277,)


## 3. Tokenization & Splitting

In [5]:
# Tokenization for Scheme 1 & 2 (Embedding Models)
VOCAB_SIZE = 10000
MAX_LENGTH = 100
EMBEDDING_DIM = 128 # Increased from 16/100 for better capacity

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X)

# Prepare Sequences for Scheme 1 (80/20)
seq_train_80 = tokenizer.texts_to_sequences(X_train_80)
seq_test_80 = tokenizer.texts_to_sequences(X_test_80)
pad_train_80 = pad_sequences(seq_train_80, maxlen=MAX_LENGTH, padding='post', truncating='post')
pad_test_80 = pad_sequences(seq_test_80, maxlen=MAX_LENGTH, padding='post', truncating='post')

# Prepare Sequences for Scheme 2 (90/10)
seq_train_90 = tokenizer.texts_to_sequences(X_train_90)
seq_test_90 = tokenizer.texts_to_sequences(X_test_90)
pad_train_90 = pad_sequences(seq_train_90, maxlen=MAX_LENGTH, padding='post', truncating='post')
pad_test_90 = pad_sequences(seq_test_90, maxlen=MAX_LENGTH, padding='post', truncating='post')


In [6]:
# Force CPU for TextVectorization to avoid GPU OOM
with tf.device('/CPU:0'):
    # Preprocessing for Scheme 3 (TF-IDF)
    vectorize_layer = TextVectorization(
        max_tokens=VOCAB_SIZE,
        output_mode='tf-idf',
        ngrams=2 # improved features
    )
    # Adapt on 80% train data
    vectorize_layer.adapt(X_train_80)
    
    tfidf_train_80 = vectorize_layer(X_train_80)
    tfidf_test_80 = vectorize_layer(X_test_80)


I0000 00:00:1770662783.113050  466161 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2337 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


## 4. Experiment 1: Bi-LSTM Model

In [7]:
# Scheme 1 (Baseline): Bi-LSTM + Embedding + 80/20 Split
tf.random.set_seed(42)
model1 = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    Bidirectional(LSTM(64, return_sequences=True)),
    Bidirectional(LSTM(32)),
    Dropout(0.5),
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(3, activation='softmax')
], name='Scheme1_BiLSTM')

model1.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model1.summary()

history1 = model1.fit(
    pad_train_80, y_train_80,
    epochs=50,
    validation_data=(pad_test_80, y_test_80),
    batch_size=32,
    callbacks=get_callbacks('scheme1_bilstm')
)

/home/asephs/Pijak/Analisis Sentimen/venv_tf/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "Scheme1_BiLSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50


2026-02-10 01:46:36.177976: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900


819/819 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7961 - loss: 0.5991
Epoch 1: val_accuracy improved from None to 0.85579, saving model to scheme1_bilstm_best.keras

Epoch 1: finished saving model to scheme1_bilstm_best.keras
819/819 ━━━━━━━━━━━━━━━━━━━━ 18s 17ms/step - accuracy: 0.8351 - loss: 0.4993 - val_accuracy: 0.8558 - val_loss: 0.4239 - learning_rate: 0.0010
Epoch 2/50
816/819 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8728 - loss: 0.3861
Epoch 2: val_accuracy did not improve from 0.85579
819/819 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step - accuracy: 0.8793 - loss: 0.3669 - val_accuracy: 0.8540 - val_loss: 0.4406 - learning_rate: 0.0010
Epoch 3/50
819/819 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8907 - loss: 0.3271
Epoch 3: val_accuracy did not improve from 0.85579
819/819 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step - accuracy: 0.8945 - loss: 0.3136 - val_accuracy: 0.8501 - val_loss: 0.5142 - learning_rate: 0.0010
Epoch 4/50
818/819 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy

## 5. Experiment 2: GRU Model

In [8]:
# Scheme 2 (Data Variation): GRU + Embedding + 90/10 Split
tf.random.set_seed(42)
model2 = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    GRU(128, return_sequences=True),
    Dropout(0.5),
    GRU(64),
    Dropout(0.5),
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    Dense(3, activation='softmax')
], name='Scheme2_GRU_90_10')

model2.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model2.summary()

history2 = model2.fit(
    pad_train_90, y_train_90,
    epochs=50,
    validation_data=(pad_test_90, y_test_90),
    batch_size=32,
    callbacks=get_callbacks('scheme2_gru')
)

Model: "Scheme2_GRU_90_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
918/922 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7487 - loss: 0.7518
Epoch 1: val_accuracy improved from None to 0.74825, saving model to scheme2_gru_best.keras

Epoch 1: finished saving model to scheme2_gru_best.keras
922/922 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - accuracy: 0.7481 - loss: 0.7287 - val_accuracy: 0.7482 - val_loss: 0.7015 - learning_rate: 0.0010
Epoch 2/50
918/922 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7926 - loss: 0.5833
Epoch 2: val_accuracy improved from 0.74825 to 0.85566, saving model to scheme2_gru_best.keras

Epoch 2: finished saving model to scheme2_gru_best.keras
922/922 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - accuracy: 0.8278 - loss: 0.4856 - val_accuracy: 0.8557 - val_loss: 0.4050 - learning_rate: 0.0010
Epoch 3/50
917/922 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8721 - loss: 0.3643
Epoch 3: val_accuracy improved from 0.85566 to 0.85810, saving model to scheme2_gru_best.keras

Epoch 3: finished saving model to scheme2_gru_best.ker

KeyboardInterrupt: 

## 6. Experiment 3: Simple NN (Baseline)

In [ ]:
# Scheme 3 (Feature Variation): Dense + TF-IDF + 80/20 Split
tf.random.set_seed(42)
model3 = Sequential([
    Input(shape=(VOCAB_SIZE,)), # TF-IDF input
    Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(3, activation='softmax')
], name='Scheme3_Dense_TFIDF')

model3.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model3.summary()

history3 = model3.fit(
    tfidf_train_80, y_train_80,
    epochs=50,
    validation_data=(tfidf_test_80, y_test_80),
    batch_size=32,
    callbacks=get_callbacks('scheme3_tfidf')
)

## 7. Evaluation & Inference

In [ ]:
def evaluate_model(model, X_test, y_test, name):
    print(f"--- Evaluation: {name} ---")
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"Test Accuracy: {acc*100:.2f}%")
    y_pred = np.argmax(model.predict(X_test), axis=1)
    print(classification_report(y_test, y_pred, target_names=['Negative', 'Neutral', 'Positive']))
    print("-"*30 + "\n")

# Evaluate all 3 schemes
evaluate_model(model1, pad_test_80, y_test_80, "Scheme 1 (Bi-LSTM, 80/20)")
evaluate_model(model2, pad_test_90, y_test_90, "Scheme 2 (GRU, 90/10)")
evaluate_model(model3, tfidf_test_80, y_test_80, "Scheme 3 (TF-IDF, 80/20)")

In [ ]:
# Inference Function
def predict_sentiment(text):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LENGTH, padding=PADDING_TYPE, truncating=TRUNC_TYPE)
    pred = model1.predict(padded)
    label_idx = np.argmax(pred)
    
    mapping = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    return mapping[label_idx]

# Test
print(predict_sentiment("This game is super fun!"))
print(predict_sentiment("Too many bugs, unplayable."))